# Unified Article Analysis Model

This notebook integrates all 6 models into a single unified system that takes an article and predicts:
1. **News Coverage** - Topic classification
2. **Intent** - Communication intent (inform, persuade, entertain, deceive)
3. **Sensationalism** - Whether content is sensational or neutral
4. **Sentiment** - Emotional sentiment (positive, negative, neutral)
5. **Reputation** - Reputation level (low, medium, high)
6. **Stance** - Political stance (against, neutral, favor)

## Usage:
```python
results = analyze_complete_article(article_text)
print(results)
```


In [1]:
# pip install pandas numpy scikit-learn torch transformers nrclex vaderSentiment
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt_tab to /home/yal149/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /home/yal149/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/yal149/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
# Core imports
import pandas as pd
import numpy as np
import re
import csv
import torch
import warnings
warnings.filterwarnings("ignore")

# ML and NLP imports
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import normalize, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import LatentDirichletAllocation

# Transformers
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# Sentiment and emotion analysis
from nrclex import NRCLex
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print("All imports loaded successfully!")

All imports loaded successfully!


In [3]:
# Data loading and preprocessing functions
COLS = ["id","label","statement","subjects","speaker","job_title",
        "state_info","party_affiliation","barely_true_cnt","false_cnt",
        "half_true_cnt","mostly_true_cnt","pants_on_fire_cnt","context","justification"]

def read_tsv(path):
    """Load TSV data with proper handling of quotes and escape characters"""
    return pd.read_csv(path, sep="\t", header=None, names=COLS,
                       engine="python", quoting=csv.QUOTE_NONE, escapechar="\\",
                       on_bad_lines="skip")

def text_of(r):
    """Combine statement, context, and justification into single text"""
    return " ".join([str(r.get("statement","")), str(r.get("context","")), str(r.get("justification",""))]).strip()

def first_subject(s):
    """Extract first subject from subjects field"""
    parts = re.split(r"[;,]", s) if isinstance(s,str) else []
    return parts[0].strip().lower() if parts and parts[0].strip() else "unknown"

print("Data loading functions defined!")

Data loading functions defined!


In [4]:
# Model 1: News Coverage Classification
print("Loading News Coverage Model...")

# Load training data
df_tr = read_tsv("train2.tsv")
df_va = read_tsv("valid.tsv")

X_tr = df_tr.apply(text_of, axis=1)
X_va = df_va.apply(text_of, axis=1)

y_tr = df_tr["subjects"].apply(first_subject)
y_va = df_va["subjects"].apply(first_subject)

keep = y_tr.ne("unknown")
X_tr, y_tr = X_tr[keep], y_tr[keep]

classes = np.unique(y_tr)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
wmap = {c:w for c,w in zip(classes, weights)}

# Train news coverage pipeline
news_coverage_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode",
                              analyzer="word", ngram_range=(1,2),
                              min_df=2, max_df=0.9, sublinear_tf=True)),
    ("clf", LinearSVC(class_weight=wmap, random_state=42))
])

news_coverage_pipe.fit(X_tr, y_tr)
pred = news_coverage_pipe.predict(X_va)
print(f"News Coverage Model - Accuracy: {accuracy_score(y_va, pred):.4f}")

def predict_news_coverage(text):
    """Predict news coverage topic for given text"""
    return news_coverage_pipe.predict([text])[0]


Loading News Coverage Model...
News Coverage Model - Accuracy: 0.3279


In [5]:
# Model 2: Intent Classification
print("Loading Intent Classification Model...")

# Load and prepare data
df = read_tsv("train2.tsv")
for c in ["statement","context","justification"]:
    df[c] = df[c].fillna("").astype(str).str.strip()

texts = df.apply(text_of, axis=1).tolist()

# Train TF-IDF vectorizer
intent_tfidf = TfidfVectorizer(
    lowercase=True, strip_accents="unicode",
    analyzer="word", ngram_range=(1,2),
    min_df=3, max_df=0.95, sublinear_tf=True
)
X = intent_tfidf.fit_transform(texts)

# Define prototypes for each intent
PROTOS = {
  "inform":   ["Officials said the department released a report with data and timelines."],
  "persuade": ["We should support this policy because it will improve outcomes."],
  "entertain":["The comedian joked about daily life in a lighthearted, playful tone."],
  "deceive":  ["You won't believe this miracle cure doctors hate; click to see the secret."]
}
CLASS_NAMES = ["inform","persuade","entertain","deceive"]

# Create prototype matrix
proto_rows = []
for name in CLASS_NAMES:
    pv = intent_tfidf.transform(PROTOS[name])
    pv_mean = np.asarray(pv.mean(axis=0))
    pv_norm = normalize(pv_mean, norm="l2", axis=1)
    proto_rows.append(pv_norm.ravel())
PROTO_MAT = np.vstack(proto_rows)

def predict_intent(title="", body=""):
    """Predict intent using prototype matching"""
    text = (" ".join([str(title or ""), str(body or "")])).strip()
    z = intent_tfidf.transform([text])
    zn = normalize(z, norm="l2", axis=1)
    scores = (zn @ PROTO_MAT.T).ravel()
    by_label = {CLASS_NAMES[i]: float(scores[i]) for i in range(4)}
    top_label = max(by_label, key=by_label.get)
    return top_label, by_label

print("Intent Classification Model loaded!")

Loading Intent Classification Model...
Intent Classification Model loaded!


In [6]:
# Model 3: Sensationalism Detection
print("Loading Sensationalism Model...")

SUPERLATIVES = {
    "shocking","unbelievable","jaw-dropping","incredible","huge","massive","disaster",
    "catastrophic","explosive","exposed","secret","ultimate","never-before-seen",
    "worst","best","always","never","everyone","no one","guaranteed","must-see"
}
HYPERBOLE = {"bombshell","meltdown","nightmare","scandal","apocalypse","panic","chaos"}
TOKEN_RE = re.compile(r"[A-Za-z]+")

def safe_cap(x, lo=0.0, hi=0.2):
    return float(np.clip(x, lo, hi))

def punct_intensity(text: str):
    s = str(text)
    denom = max(1, len(s))
    ex = s.count("!")
    qm = s.count("?")
    return ex/denom, qm/denom

def all_caps_ratio(text):
    toks_caps = re.findall(r"\b[A-Z]{3,}\b", str(text))
    denom = max(1, len(TOKEN_RE.findall(str(text))))
    return len(toks_caps) / denom

def lex_density(text, vocab):
    toks = TOKEN_RE.findall(str(text))
    return sum(t in vocab for t in toks) / max(1, len(toks))

EVIDENCE_PATTERNS = [
    r"\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b",
    r"\b(19|20)\d{2}\b",
    r"%",
    r"https?://",
    r"\"[^\"]+\"",
    r"\baccording to\b",
    r"\breport(ed|s)? by\b|\bstudy\b|\bsurvey\b"
]
EVIDENCE_RE = [re.compile(pat, flags=re.I) for pat in EVIDENCE_PATTERNS]

def evidence_anchors(text):
    s = str(text)
    hits = sum(len(r.findall(s)) for r in EVIDENCE_RE)
    toks = max(1, len(TOKEN_RE.findall(s)))
    dens = np.log1p(hits) / max(10, toks)
    return float(np.clip(dens, 0.0, 0.2))

def vader_compound(s):
    analyzer = SentimentIntensityAnalyzer()
    return float(analyzer.polarity_scores(str(s))["compound"])

def extract_sensationalism_features(row):
    statement = row.get("statement_raw", "") or ""
    justification = row.get("justification", "") or ""

    ex_rate, qm_rate = punct_intensity(statement)
    caps_ratio = all_caps_ratio(statement)
    superl_density = lex_density(statement, SUPERLATIVES | HYPERBOLE)

    comp_stmt = vader_compound(statement)
    comp_just = vader_compound(justification) if str(justification).strip() else comp_stmt
    mismatch_delta = float(comp_stmt - comp_just)

    if not str(justification).strip():
        mismatch_delta *= 0.2

    anchors_stmt = evidence_anchors(statement)
    anchors_just = evidence_anchors(justification)
    anchors = max(anchors_stmt, anchors_just)

    ex_rate = safe_cap(ex_rate)
    qm_rate = safe_cap(qm_rate)
    caps_ratio = safe_cap(caps_ratio)
    superl_density = safe_cap(superl_density)

    return {
        "exclam_rate": ex_rate,
        "all_caps_ratio": caps_ratio,
        "hyperbole_density": superl_density,
        "headline_compound": float(comp_stmt),
        "justification_compound": float(comp_just),
        "mismatch_delta": mismatch_delta,
        "evidence_anchors": float(anchors),
    }

def build_sensationalism_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    feats = [extract_sensationalism_features(r) for r in df.to_dict("records")]
    return pd.DataFrame(feats, index=df.index).astype("float32")

# Train sensationalism model
train_df = df_tr.copy()
X_sens = build_sensationalism_feature_frame(train_df)
FEATURE_ORDER = X_sens.columns.tolist()
COL2IDX = {name: i for i, name in enumerate(FEATURE_ORDER)}

X_sens["intensity"] = (
    X_sens["hyperbole_density"] +
    X_sens["exclam_rate"] +
    X_sens["all_caps_ratio"] +
    0.5 * np.abs(X_sens["headline_compound"])
)

q_intense = X_sens["intensity"].quantile(0.85)
q_support = X_sens["evidence_anchors"].quantile(0.50)

X_sens["mismatch_delta_pos"] = np.clip(X_sens["mismatch_delta"], 0, 1)
sens_rule = ((X_sens["intensity"] >= q_intense) & (X_sens["evidence_anchors"] <= q_support)) | (X_sens["mismatch_delta_pos"] >= 0.3)
y_silver = sens_rule.astype(int)

base_lr = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
sensationalism_clf = CalibratedClassifierCV(base_lr, method="isotonic", cv=3)
sensationalism_clf.fit(X_sens[FEATURE_ORDER].values, y_silver.values)

def _features_for_inference(statement: str, justification: str = "") -> np.ndarray:
    row = pd.DataFrame([{"statement_raw": statement, "justification": justification}])
    base = build_sensationalism_feature_frame(row)
    base["intensity"] = (
        base["hyperbole_density"] + base["exclam_rate"] + base["all_caps_ratio"]
        + 0.5 * (np.abs(base["headline_compound"]))
    )
    base["mismatch_delta_pos"] = np.clip(base["mismatch_delta"], 0, 1)
    aligned = base.reindex(columns=FEATURE_ORDER, fill_value=0.0)
    return aligned.values[0]

def predict_sensationalism(statement: str, justification: str = ""):
    f_vec = _features_for_inference(statement, justification)
    p_sens = float(sensationalism_clf.predict_proba([f_vec])[0, 1])
    score_0_10 = float(np.clip(10.0 * (1.0 - p_sens), 0.0, 10.0))
    label = "sensational" if p_sens >= 0.5 else "neutral"
    confidence = float(max(p_sens, 1 - p_sens))

    headline_intensity = float(
        f_vec[COL2IDX["hyperbole_density"]]
        + f_vec[COL2IDX["exclam_rate"]]
        + f_vec[COL2IDX["all_caps_ratio"]]
    )
    emotional_charge = float(abs(f_vec[COL2IDX["headline_compound"]]))
    evidence = float(f_vec[COL2IDX["evidence_anchors"]])
    mismatch = float(f_vec[COL2IDX["mismatch_delta"]])

    return {
        "factor": "sensationalism",
        "score": round(score_0_10, 3),
        "confidence": round(confidence, 3),
        "label": label,
        "subscores": {
            "headline_intensity": headline_intensity,
            "emotional_charge": emotional_charge,
            "evidence_anchors": evidence,
            "mismatch_delta": mismatch,
        },
    }

print("Sensationalism Model loaded!")

Loading Sensationalism Model...
Sensationalism Model loaded!


In [7]:
# Model 4: Sentiment Analysis
print("Loading Sentiment Analysis Model...")

# Sentiment analysis constants
ALPHA = 1.0
EPS = 1e-8
EMOTIONS_TO_ANALYZE = ["positive", "negative", "anger", "fear", "disgust",
                       "sadness", "joy", "anticipation", "trust", "surprise"]
_token_re = re.compile(r"[A-Za-z']+")

def _word_count(text):
    return len(_token_re.findall(str(text)))

def nrc_doc_score_from_text(text, alpha: float = ALPHA):
    emo = NRCLex(str(text))
    pos = emo.raw_emotion_scores.get('positive', 0)
    neg = emo.raw_emotion_scores.get('negative', 0)
    return float(np.log((pos + alpha) / (neg + alpha)))

def get_sentiment_class(vader_row):
    c = float(vader_row.get("compound", 0.0))
    if c >= 0.05:
        return 'Positive'
    elif c <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

class ModelArtifacts:
    def __init__(self, vectorizer, topic_model, topic_mu, topic_sigma):
        self.vectorizer = vectorizer
        self.topic_model = topic_model
        self.topic_mu = topic_mu
        self.topic_sigma = topic_sigma

def train_sentiment_models(train_df, text_col="statement"):
    train_texts = train_df[text_col].fillna("").astype(str).tolist()

    vectorizer = CountVectorizer(max_df=0.9, min_df=5, stop_words='english')
    X_train = vectorizer.fit_transform(train_texts)

    topic_model = LatentDirichletAllocation(n_components=20, random_state=42, learning_method="batch")
    topic_model.fit(X_train)

    theta_train = topic_model.transform(X_train)
    s_train = np.array([nrc_doc_score_from_text(s) for s in train_texts])

    weights_sum = theta_train.sum(axis=0) + EPS
    topic_mu = (theta_train.T @ s_train) / weights_sum

    diffs = s_train[:, None] - topic_mu[None, :]
    topic_var = (theta_train * diffs ** 2).sum(axis=0) / weights_sum
    topic_sigma = np.sqrt(topic_var + EPS)

    return ModelArtifacts(
        vectorizer=vectorizer,
        topic_model=topic_model,
        topic_mu=topic_mu,
        topic_sigma=topic_sigma
    )

def extract_sentiment_features_from_statement(statement: str, artifacts, analyzer) -> dict:
    s = "" if statement is None else str(statement)
    wc = _word_count(s)

    emo_obj = NRCLex(s)
    emo_counts = {e: int(emo_obj.raw_emotion_scores.get(e, 0)) for e in EMOTIONS_TO_ANALYZE}
    emo_densities = {f"{e}_density": (emo_counts[e] / wc) if wc > 0 else 0.0 for e in EMOTIONS_TO_ANALYZE}

    emotion_logratio = float(np.log((emo_counts["positive"] + ALPHA) / (emo_counts["negative"] + ALPHA)))
    s_val = emotion_logratio

    X = artifacts.vectorizer.transform([s])
    theta = artifacts.topic_model.transform(X)

    mu_hat = float((theta @ artifacts.topic_mu)[0])
    var_hat = float((theta @ (artifacts.topic_sigma ** 2))[0] + EPS)
    sd_hat = float(np.sqrt(var_hat))

    sent_dev_diff = float(s_val - mu_hat)
    sent_dev_z = float(sent_dev_diff / sd_hat) if sd_hat > 0 else 0.0

    vader = analyzer.polarity_scores(s)
    sentiment_value = get_sentiment_class(vader)
    sentiment_extremity = abs(float(vader.get("compound", 0.0)))

    return {
        "word_count": wc,
        "emotion_logratio": float(emotion_logratio),
        "sent_dev_diff": float(sent_dev_diff),
        "sent_dev_z": float(sent_dev_z),
        "vader_pos": float(vader.get("pos", 0.0)),
        "vader_neu": float(vader.get("neu", 0.0)),
        "vader_neg": float(vader.get("neg", 0.0)),
        "vader_compound": float(vader.get("compound", 0.0)),
        "sentiment_value": sentiment_value,
        "sentiment_extremity": float(sentiment_extremity),
    }

# Train sentiment model
sentiment_artifacts = train_sentiment_models(df_tr)
vader_analyzer = SentimentIntensityAnalyzer()

def create_sentiment_feature_dataframe(df: pd.DataFrame,
                             artifacts: ModelArtifacts,
                             analyzer: SentimentIntensityAnalyzer,
                             text_col="statement"):

    feature_rows = df[text_col].apply(
        extract_sentiment_features_from_statement,
        args=(artifacts, analyzer)
    )

    features_df = pd.DataFrame(feature_rows.tolist())
    return features_df

train_sentiment_features = create_sentiment_feature_dataframe(df_tr, sentiment_artifacts, vader_analyzer)
validation_sentiment_features = create_sentiment_feature_dataframe(df_va, sentiment_artifacts, vader_analyzer)

features = ["emotion_logratio", "sent_dev_z"]

X_tv = pd.concat([train_sentiment_features[features], validation_sentiment_features[features]], axis=0)
y_tr_sent = train_sentiment_features["sentiment_value"]
y_va_sent = validation_sentiment_features["sentiment_value"]
y_tv = pd.concat([y_tr_sent, y_va_sent], axis=0)

sentiment_pipe = Pipeline(steps=[
    ("scale", StandardScaler()),
    ("clf", MLPClassifier(alpha=1, max_iter=1000, random_state=42)),
])
sentiment_pipe.fit(X_tv, y_tv)

def predict_sentiment(statement):
    """Predict sentiment for given statement"""
    feature_dict = extract_sentiment_features_from_statement(statement, sentiment_artifacts, vader_analyzer)
    features_df = pd.DataFrame([feature_dict])
    X_to_predict = features_df[features]
    y_pred = sentiment_pipe.predict(X_to_predict)
    return y_pred[0]

print("Sentiment Analysis Model loaded!")

Loading Sentiment Analysis Model...
Sentiment Analysis Model loaded!


In [8]:
# Model 5: Reputation Classification
print("Loading Reputation Model...")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load reputation model
try:
    reputation_model_path = "./reputation_model"
    reputation_model = DistilBertForSequenceClassification.from_pretrained(reputation_model_path)
    reputation_tokenizer = DistilBertTokenizerFast.from_pretrained(reputation_model_path)
    reputation_model.to(device)
    
    # Label encoder for reputation
    rep_encoder = LabelEncoder()
    rep_encoder.fit(["low", "medium", "high"])
    
    print("Reputation Model loaded successfully!")
    
except Exception as e:
    print(f" Reputation model not found: {e}")
    print("Using fallback reputation prediction...")
    
    # Fallback reputation model
    reputation_model = None
    reputation_tokenizer = None
    rep_encoder = None


Loading Reputation Model...
Using device: cpu
 Reputation model not found: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory ./reputation_model.
Using fallback reputation prediction...


In [9]:
# Model 6: Stance Classification
print("Loading Stance Model...")

# Ensure imports are available
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

try:
    stance_model_path = "./stance_model"
    stance_tokenizer = AutoTokenizer.from_pretrained(stance_model_path)
    stance_model = AutoModelForSequenceClassification.from_pretrained(stance_model_path)
    stance_model.to(device)
    
    # Stance mapping
    id2stance = {0: "support", 1: "deny", 2: "neutral"}
    
    print(" Stance Model loaded successfully!")
    
except Exception as e:
    print(f" Stance model not found: {e}")
    print("Using fallback stance prediction...")
    
    # Fallback stance model
    stance_model = None
    stance_tokenizer = None
    id2stance = None


Loading Stance Model...
Using device: cpu
 Stance model not found: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory ./stance_model.
Using fallback stance prediction...


In [10]:
import nltk
from collections import Counter

def predict_article_reputation(article_text=None, sentences=None):
    """
    Predicts reputation for each sentence and aggregates results with majority vote.
    Can accept either full article text or pre-split sentences.
    """
    # Check if model is loaded
    if reputation_model is None or reputation_tokenizer is None:
        return {"final_label": "medium", "counts": {"medium": 1}}
    
    if sentences is None:
        if article_text is None:
            return {"final_label": "medium", "counts": {"medium": 1}}
        # Split article into sentences if not provided
        sentences = nltk.sent_tokenize(article_text)
        sentences = [s.strip() for s in sentences if s.strip()]
    
    if not sentences:
        return {"final_label": "medium", "counts": {"medium": 1}}

    results = []
    for sent in sentences:
        # Tokenize each sentence separately
        inputs = reputation_tokenizer(
            sent,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        # Forward pass
        with torch.no_grad():
            outputs = reputation_model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            pred_id = torch.argmax(probs, dim=-1).item()
            label = reputation_model.config.id2label[pred_id]
            score = probs[0][pred_id].item()

        results.append((sent, label, score))

    # --- Majority vote across sentences ---
    labels = [label for _, label, _ in results]
    label_counts = Counter(labels)
    majority_label = label_counts.most_common(1)[0][0]
    reputation_model.config.id2label = {0: "low", 1: "medium", 2: "high"}
    reputation_model.config.label2id = {"low": 0, "medium": 1, "high": 2}
    return {
        "final_label": majority_label,
        "counts": dict(label_counts),
    }

In [11]:
import nltk
from collections import Counter

def predict_article_stance(article_text=None, sentences=None):
    """
    Predicts stance for each sentence and aggregates results with majority vote.
    Can accept either full article text or pre-split sentences.
    """
    # Check if model is loaded
    if stance_model is None or stance_tokenizer is None:
        return {"final_label": "neutral", "counts": {"neutral": 1}}
    
    if sentences is None:
        if article_text is None:
            return {"final_label": "neutral", "counts": {"neutral": 1}}
        # Split article into sentences if not provided
        sentences = nltk.sent_tokenize(article_text)
        sentences = [s.strip() for s in sentences if s.strip()]
    
    if not sentences:
        return {"final_label": "neutral", "counts": {"neutral": 1}}

    results = []
    for sent in sentences:
        # Tokenize each sentence separately
        inputs = stance_tokenizer(
            sent,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        # Forward pass
        with torch.no_grad():
            outputs = stance_model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            pred_id = torch.argmax(probs, dim=-1).item()
            label = stance_model.config.id2label[pred_id]
            score = probs[0][pred_id].item()

        results.append((sent, label, score))

    # --- Majority vote across sentences ---
    labels = [label for _, label, _ in results]
    label_counts = Counter(labels)
    majority_label = label_counts.most_common(1)[0][0]

    return {
        "final_label": majority_label,
        "counts": dict(label_counts),
    }

In [12]:
# Sentence-level Article Analysis Function
def analyze_complete_article(article_text):
    """
    Complete article analysis using sentence-level voting from all 6 models
    
    Args:
        article_text (str): The full article text
    
    Returns:
        dict: Dictionary containing all model predictions with sentence-level details
    """
    import re
    import nltk
    from collections import Counter
    
    results = {
        # "article_text": article_text[:200] + "..." if len(article_text) > 200 else article_text,
        # "article_length": len(article_text),
        "predictions": {},
        "sentence_analysis": {}
    }
    
    # Split article into sentences ONCE using nltk for better accuracy
    sentences = nltk.sent_tokenize(article_text)
    sentences = [s.strip() for s in sentences if s.strip() and len(s.strip()) > 10]
    
    results["num_sentences"] = len(sentences)
    results["sentences"] = sentences
    
    # Initialize counters for voting
    news_coverage_votes = []
    intent_votes = []
    sensationalism_votes = []
    sentiment_votes = []
    reputation_votes = []
    stance_votes = []
    
    # Process each sentence
    sentence_results = []
    
    for i, sentence in enumerate(sentences):
        sentence_result = {
            "sentence_id": i,
            "sentence_text": sentence,
            "predictions": {}
        }
        
        # 1. News Coverage Prediction
        try:
            news_coverage = predict_news_coverage(sentence)
            news_coverage_votes.append(news_coverage)
            sentence_result["predictions"]["news_coverage"] = news_coverage
        except Exception as e:
            sentence_result["predictions"]["news_coverage"] = f"error: {str(e)}"
        
        # 2. Intent Prediction
        try:
            intent, intent_scores = predict_intent("", sentence)
            intent_votes.append(intent)
            sentence_result["predictions"]["intent"] = {
                "primary_intent": intent,
                "confidence_scores": intent_scores
            }
        except Exception as e:
            sentence_result["predictions"]["intent"] = f"error: {str(e)}"
        
        # 3. Sensationalism Prediction
        try:
            sensationalism_result = predict_sensationalism(sentence)
            sensationalism_votes.append(sensationalism_result["label"])
            sentence_result["predictions"]["sensationalism"] = sensationalism_result
        except Exception as e:
            sentence_result["predictions"]["sensationalism"] = f"error: {str(e)}"
        
        # 4. Sentiment Prediction
        try:
            sentiment = predict_sentiment(sentence)
            sentiment_votes.append(sentiment)
            sentence_result["predictions"]["sentiment"] = sentiment
        except Exception as e:
            sentence_result["predictions"]["sentiment"] = f"error: {str(e)}"
        
        # 5. Reputation Prediction (use pre-split sentences for transformer models)
        try:
            if i == 0:  # Only predict once for reputation (uses pre-split sentences)
                reputation_result = predict_article_reputation(sentences=sentences)
                reputation = reputation_result["final_label"]
                reputation_votes.append(reputation)
                sentence_result["predictions"]["reputation"] = {
                    "level": reputation,
                    "counts": reputation_result.get("counts", {})
                }
            else:
                sentence_result["predictions"]["reputation"] = "same as first sentence"
        except Exception as e:
            sentence_result["predictions"]["reputation"] = f"error: {str(e)}"
        
        # 6. Stance Prediction (use pre-split sentences for transformer models)
        try:
            if i == 0:  # Only predict once for stance (uses pre-split sentences)
                stance_result = predict_article_stance(sentences=sentences)
                stance = stance_result["final_label"]
                stance_votes.append(stance)
                sentence_result["predictions"]["stance"] = {
                    "stance": stance,
                    "counts": stance_result.get("counts", {})
                }
            else:
                sentence_result["predictions"]["stance"] = "same as first sentence"
        except Exception as e:
            sentence_result["predictions"]["stance"] = f"error: {str(e)}"
        
        sentence_results.append(sentence_result)
    
    results["sentence_analysis"] = sentence_results
    
    # Vote aggregation for article-level predictions
    try:
        # 1. News Coverage - Majority Vote
        news_coverage_counter = Counter(news_coverage_votes)
        article_news_coverage = news_coverage_counter.most_common(1)[0][0]
        results["predictions"]["news_coverage"] = {
            "topic": article_news_coverage,
            "vote_distribution": dict(news_coverage_counter),
            "model": "LinearSVC + TF-IDF (Majority Vote)"
        }
    except Exception as e:
        results["predictions"]["news_coverage"] = {"error": str(e)}
    
    try:
        # 2. Intent - Majority Vote
        intent_counter = Counter(intent_votes)
        article_intent = intent_counter.most_common(1)[0][0]
        results["predictions"]["intent"] = {
            "primary_intent": article_intent,
            "vote_distribution": dict(intent_counter),
            "model": "Prototype Matching (Majority Vote)"
        }
    except Exception as e:
        results["predictions"]["intent"] = {"error": str(e)}
    
    try:
        # 3. Sensationalism - Majority Vote
        sensationalism_counter = Counter(sensationalism_votes)
        article_sensationalism = sensationalism_counter.most_common(1)[0][0]
        results["predictions"]["sensationalism"] = {
            "label": article_sensationalism,
            "vote_distribution": dict(sensationalism_counter),
            "model": "RandomForest (Majority Vote)"
        }
    except Exception as e:
        results["predictions"]["sensationalism"] = {"error": str(e)}
    
    try:
        # 4. Sentiment - Majority Vote
        sentiment_counter = Counter(sentiment_votes)
        article_sentiment = sentiment_counter.most_common(1)[0][0]
        results["predictions"]["sentiment"] = {
            "sentiment": article_sentiment,
            "vote_distribution": dict(sentiment_counter),
            "model": "MLPClassifier (Majority Vote)"
        }
    except Exception as e:
        results["predictions"]["sentiment"] = {"error": str(e)}
    
    try:
        # 5. Reputation - Single prediction (transformer model)
        if reputation_votes:
            reputation_result = predict_article_reputation(sentences=sentences)
            results["predictions"]["reputation"] = {
                "level": reputation_result["final_label"],
                "counts": reputation_result.get("counts", {}),
                "model": "DistilBERT (Sentence-level Majority Vote)"
            }
        else:
            results["predictions"]["reputation"] = {"error": "No prediction"}
    except Exception as e:
        results["predictions"]["reputation"] = {"error": str(e)}
    
    try:
        # 6. Stance - Single prediction (transformer model)
        if stance_votes:
            stance_result = predict_article_stance(sentences=sentences)
            results["predictions"]["stance"] = {
                "stance": stance_result["final_label"],
                "counts": stance_result.get("counts", {}),
                "model": "AutoModel (Sentence-level Majority Vote)"
            }
        else:
            results["predictions"]["stance"] = {"error": "No prediction"}
    except Exception as e:
        results["predictions"]["stance"] = {"error": str(e)}
    
    return results

def print_analysis_results(results):
    """Print formatted analysis results - only final predictions with voting stats"""
    # print("=" * 80)
    # print("ARTICLE ANALYSIS RESULTS (MAJORITY VOTE)")
    # print("=" * 80)
    # print(f"Article: {results['article_text']}")
    # print(f"Length: {results['article_length']} characters")
    print(f"Number of sentences: {results['num_sentences']}")
    # print()
    
    print("FINAL PREDICTIONS:")
    print("-" * 30)
    
    for model_name, prediction in results["predictions"].items():
        if "error" not in prediction:
            if model_name == "news_coverage":
                print(f"News Topic: {prediction['topic']}")
                if "vote_distribution" in prediction:
                    print(f"   Vote Count: {prediction['vote_distribution']}")
            elif model_name == "intent":
                print(f"Intent: {prediction['primary_intent']}")
                if "vote_distribution" in prediction:
                    print(f"   Vote Count: {prediction['vote_distribution']}")
            elif model_name == "sensationalism":
                print(f" Sensationalism: {prediction['label']}")
                if "vote_distribution" in prediction:
                    print(f"   Vote Count: {prediction['vote_distribution']}")
            elif model_name == "sentiment":
                print(f" Sentiment: {prediction['sentiment']}")
                if "vote_distribution" in prediction:
                    print(f"   Vote Count: {prediction['vote_distribution']}")
            elif model_name == "reputation":
                print(f" Reputation: {prediction['level']}")
                if "counts" in prediction and prediction["counts"]:
                    print(f"   Sentence Votes: {prediction['counts']}")
            elif model_name == "stance":
                print(f" Stance: {prediction['stance']}")
                if "counts" in prediction and prediction["counts"]:
                    print(f"   Sentence Votes: {prediction['counts']}")
        else:
            print(f" {model_name}: Error - {prediction['error']}")
        print()
    
    print("=" * 80)

print("Unified analysis function created!")

Unified analysis function created!


In [13]:
# Whole Article Analysis (No Pre-processing)
def analyze_whole_article(article_text):
    """
    Complete article analysis treating the article as a single long sentence.
    No sentence splitting or pre-processing - analyzes the entire article at once.
    
    Args:
        article_text (str): The full article text
    
    Returns:
        dict: Dictionary containing all model predictions for the whole article
    """
    import re
    from collections import Counter
    
    results = {
        "article_length": len(article_text),
        "analysis_type": "whole_article",
        "predictions": {}
    }
    
    print(f"📄 Analyzing whole article ({len(article_text)} characters)...")
    
    # 1. News Coverage Prediction (whole article)
    try:
        news_coverage = predict_news_coverage(article_text)
        results["predictions"]["news_coverage"] = {
            "topic": news_coverage,
            "model": "LinearSVC + TF-IDF (Whole Article)"
        }
        print(f"News Coverage: {news_coverage}")
    except Exception as e:
        results["predictions"]["news_coverage"] = {"error": str(e)}
        print(f"News Coverage Error: {str(e)}")
    
    # 2. Intent Prediction (whole article)
    try:
        intent, intent_scores = predict_intent("", article_text)
        results["predictions"]["intent"] = {
            "primary_intent": intent,
            "confidence_scores": intent_scores,
            "model": "Prototype Matching (Whole Article)"
        }
        print(f"Intent: {intent}")
    except Exception as e:
        results["predictions"]["intent"] = {"error": str(e)}
        print(f" Intent Error: {str(e)}")
    
    # 3. Sensationalism Prediction (whole article)
    try:
        sensationalism_result = predict_sensationalism(article_text)
        results["predictions"]["sensationalism"] = {
            "label": sensationalism_result["label"],
            "score": sensationalism_result["score"],
            "confidence": sensationalism_result["confidence"],
            "subscores": sensationalism_result["subscores"],
            "model": "RandomForest (Whole Article)"
        }
        print(f" Sensationalism: {sensationalism_result['label']} (score: {sensationalism_result['score']})")
    except Exception as e:
        results["predictions"]["sensationalism"] = {"error": str(e)}
        print(f" Sensationalism Error: {str(e)}")
    
    # 4. Sentiment Prediction (whole article)
    try:
        sentiment = predict_sentiment(article_text)
        results["predictions"]["sentiment"] = {
            "sentiment": sentiment,
            "model": "MLPClassifier (Whole Article)"
        }
        print(f"Sentiment: {sentiment}")
    except Exception as e:
        results["predictions"]["sentiment"] = {"error": str(e)}
        print(f" Sentiment Error: {str(e)}")
    
    # 5. Reputation Prediction (whole article - may need truncation handling)
    try:
        # For very long articles, we might need to truncate for transformer models
        max_length = 512  # Adjust based on your model's max length
        if len(article_text) > max_length * 4:  # Rough estimate for token count
            truncated_text = article_text[:max_length * 4]  # Truncate if too long
            print(f"  Article truncated for reputation analysis ({len(truncated_text)} chars)")
        else:
            truncated_text = article_text
            
        reputation_result = predict_article_reputation(article_text=truncated_text)
        results["predictions"]["reputation"] = {
            "level": reputation_result["final_label"],
            "counts": reputation_result.get("counts", {}),
            "model": "DistilBERT (Whole Article)",
            "truncated": len(truncated_text) < len(article_text)
        }
        print(f" Reputation: {reputation_result['final_label']}")
    except Exception as e:
        results["predictions"]["reputation"] = {"error": str(e)}
        print(f" Reputation Error: {str(e)}")
    
    # 6. Stance Prediction (whole article - may need truncation handling)
    try:
        # For very long articles, we might need to truncate for transformer models
        max_length = 512  # Adjust based on your model's max length
        if len(article_text) > max_length * 4:  # Rough estimate for token count
            truncated_text = article_text[:max_length * 4]  # Truncate if too long
            print(f"  Article truncated for stance analysis ({len(truncated_text)} chars)")
        else:
            truncated_text = article_text
            
        stance_result = predict_article_stance(article_text=truncated_text)
        results["predictions"]["stance"] = {
            "stance": stance_result["final_label"],
            "counts": stance_result.get("counts", {}),
            "model": "AutoModel (Whole Article)",
            "truncated": len(truncated_text) < len(article_text)
        }
        print(f" Stance: {stance_result['final_label']}")
    except Exception as e:
        results["predictions"]["stance"] = {"error": str(e)}
        print(f" Stance Error: {str(e)}")
    
    return results

def print_whole_article_results(results):
    """Print formatted results for whole article analysis"""
    print("\n" + "=" * 80)
    print("WHOLE ARTICLE ANALYSIS RESULTS")
    print("=" * 80)
    print(f"Article Length: {results['article_length']} characters")
    print(f"Analysis Type: {results['analysis_type']}")
    print()
    
    print(" PREDICTIONS:")
    print("-" * 30)
    
    for model_name, prediction in results["predictions"].items():
        if "error" not in prediction:
            if model_name == "news_coverage":
                print(f"News Topic: {prediction['topic']}")
            elif model_name == "intent":
                print(f"Intent: {prediction['primary_intent']}")
                print(f"   Confidence Scores: {prediction['confidence_scores']}")
            elif model_name == "sensationalism":
                print(f"Sensationalism: {prediction['label']} (score: {prediction['score']})")
                print(f"   Confidence: {prediction['confidence']}")
            elif model_name == "sentiment":
                print(f" Sentiment: {prediction['sentiment']}")
            elif model_name == "reputation":
                print(f"Reputation: {prediction['level']}")
                if prediction.get("truncated"):
                    print("   Article was truncated for analysis")
            elif model_name == "stance":
                print(f" Stance: {prediction['stance']}")
                if prediction.get("truncated"):
                    print("     Article was truncated for analysis")
        else:
            print(f" {model_name}: Error - {prediction['error']}")
        print()
    
    print("=" * 80)

print("Whole article analysis function created!")

Whole article analysis function created!


In [14]:
# Test Whole Article Analysis
print("Testing Whole Article Analysis (No Pre-processing)")
print("=" * 60)

# Test with a sample article
test_article = """
The government announced new economic policies today that will affect millions of citizens. Officials say the changes will improve employment rates and boost economic growth. The policy includes tax cuts for middle-class families and increased funding for education programs. Critics argue that the measures are insufficient to address the current economic challenges. The announcement comes amid rising inflation and concerns about the national debt. Supporters believe these policies will create jobs and stimulate the economy. The implementation is expected to begin next quarter with full rollout by year-end.
"""

print(f"Test Article Length: {len(test_article)} characters")
print("\n" + "="*60)

# Run whole article analysis
results = analyze_whole_article(test_article)

# Print results
print_whole_article_results(results)

print("\n🎯 Summary:")
print("-" * 20)
for model_name, prediction in results["predictions"].items():
    if "error" not in prediction:
        if model_name == "news_coverage":
            print(f" News Topic: {prediction['topic']}")
        elif model_name == "intent":
            print(f" Intent: {prediction['primary_intent']}")
        elif model_name == "sensationalism":
            print(f" Sensationalism: {prediction['label']}")
        elif model_name == "sentiment":
            print(f" Sentiment: {prediction['sentiment']}")
        elif model_name == "reputation":
            print(f" Reputation: {prediction['level']}")
        elif model_name == "stance":
            print(f"  Stance: {prediction['stance']}")

print("\n Whole article analysis test completed!")

Testing Whole Article Analysis (No Pre-processing)
Test Article Length: 615 characters

📄 Analyzing whole article (615 characters)...
News Coverage: jobs
Intent: persuade
 Sensationalism: sensational (score: 0.0)
Sentiment: Neutral
 Reputation: medium
 Stance: neutral

WHOLE ARTICLE ANALYSIS RESULTS
Article Length: 615 characters
Analysis Type: whole_article

 PREDICTIONS:
------------------------------
News Topic: jobs

Intent: persuade
   Confidence Scores: {'inform': 0.02528764761892692, 'persuade': 0.06707700108139836, 'entertain': 0.007035017789647699, 'deceive': 0.015537059327244359}

Sensationalism: sensational (score: 0.0)
   Confidence: 1.0

 Sentiment: Neutral

Reputation: medium

 Stance: neutral


🎯 Summary:
--------------------
 News Topic: jobs
 Intent: persuade
 Sensationalism: sensational
 Sentiment: Neutral
 Reputation: medium
  Stance: neutral

 Whole article analysis test completed!


In [15]:
# Test the unified model with sample articles
print("Testing Unified Article Analysis Model")
print("=" * 50)

# Test articles
test_articles = [
    "The government announced new economic policies today that will affect millions of citizens. Officials say the changes will improve employment rates and boost economic growth. The policy includes tax cuts for middle-class families and increased funding for education programs."]

# Run analysis on each test article
for i, article in enumerate(test_articles, 1):
    print(f"\n Test Article {i}:")
    results = analyze_complete_article(article)
    print_analysis_results(results)
    print()

print("All tests completed!")

Testing Unified Article Analysis Model

 Test Article 1:
Number of sentences: 3
FINAL PREDICTIONS:
------------------------------
News Topic: economy
   Vote Count: {'economy': 2, 'taxes': 1}

Intent: persuade
   Vote Count: {'persuade': 3}

 Sensationalism: sensational
   Vote Count: {'sensational': 3}

 Sentiment: Negative
   Vote Count: {'Negative': 2, 'Positive': 1}

 Reputation: medium
   Sentence Votes: {'medium': 1}

 Stance: neutral
   Sentence Votes: {'neutral': 1}


All tests completed!


In [16]:
# Usage Example
print("Usage Example:")
print("=" * 30)

# Example article
example_article = """
Skip to main contentSkip to navigationSkip to navigation
Support the Guardian
Fund the free press with $5 per month
Support us
Print subscriptions
Newsletters
Sign in
US
The Guardian - Back to homeThe Guardian

News
Opinion
Sport
Culture
Lifestyle
Show more
UK
UK politics
Education
Media
Society
Law
Scotland
Wales
Northern Ireland
Extreme closeup of an angry man's face
View image in fullscreen
‘What might happen if we didn’t default to automatic rage?’ Photograph: XiXinXing/Getty Images
Society
 This article is more than 1 year old
We’re living in the age of rage. I’m a psychoanalyst – here’s what we need to do to calm down
This article is more than 1 year old
Anger has come to define the public mood – felt in the posts of social media warriors and harnessed by populist agitators such as Trump and Farage. But why are we so mad, and how can we learn to redirect our feelings?

Josh Cohen
Sun 15 Sep 2024 03.00 EDT
Share
Every morning, my inbox heaves with a new tranche of email alerts from Nextdoor, the social networking service for neighbourhoods where people in the area post recommendations, inquiries, requests, offers, information. The tone can be chummy, jocular, kindly, anxious, but mostly the posts are angry. They include vituperative warnings about dodgy tradesmen; outraged reports of cruelty to animals witnessed by neighbours; snatches of grainy Ring camera footage purporting to show actual or attempted burglaries; complaints of junkies splayed on park benches and of predatory lone men approaching young girls; reports of vandalism, fly-tipping, charity muggers, phone scammers, poor restaurant service and late-night noise.

My heart sinks at each new set of notifications, festooned with rage emojis and opprobrium for lowlifes, SCUM, animals! Yet I’ve never been tempted to unsubscribe – and not only because the service is also a surprising showcase for human solidarity, reuniting desperate owners with their cats and wallets, offering help and advice to the hungry and infirm. Much as I appreciate these outbreaks of decency, it’s the rage that continues to draw me. A batch of Nextdoor updates is a live window on the vexations of modern urban living, an electric chorus of sighs, growls and screams from the frontline of everyday reality.


The anger on Nextdoor strikes me as different in quality from the triumphal rage that characterises so much content on X, where it can seem as though disagreement can be voiced only in tones of righteous indignation and caustic sarcasm. This is why I stay off the platform; I can’t look at it without being struck by the trolling, shaming and piling on, the atmosphere of free-form hate and fury.

The social media giants foster the grandiose illusion that your smartphone is a global megaphone, blasting out your furious convictions on the social, ethical and geopolitical dilemmas of the moment to a potential audience of millions (even if your actual followers number in the low hundreds). It cultivates a mode of anger that is both impersonal and self-important, a style of sloganising that is grindingly repetitive, each post an echo of the last.

Unrest in the UK as a man holds a smoke flare while police officers guard a hotel in Rotherham, 4 August 2024.
View image in fullscreen
Unrest in the UK as a man holds a smoke flare while police officers guard a hotel in Rotherham, 4 August 2024. Photograph: Hollie Adams/Reuters

The anger voiced in Nextdoor posts seems endearingly human in contrast. Posts will reach (and likely interest) only those in your own neighbourhood, meaning there is less incentive to engage in performative provocation. Posters speak of wanting to vomit, to scream, to cry, to punch their thieving, fly-tipping, noisy tormentors. They remind us that anger, like all significant feelings, is experienced first at a bodily level, as a pressure towards discharge through the mouth or limbs.

In his 2006 book, Rage and Time, the German philosopher Peter Sloterdijk makes a distinction between two kinds of rage that casts some light on the mood and colour of our own moment. The first kind, which Sloterdijk calls “banked” rage, refers to the rage gathered and directed by popular, sometimes revolutionary leaders who achieved power by harnessing and “banking” the rage of the “humiliated and offended” victims of injustice and oppression across generations. Such leaders seek to gather a mass of anger into a “rage bank”, a reservoir of emotional and political capital that could power a long-term transformation of society for better or worse.

For the populist agitator, the aim seems to be to stoke a rage for which there can be no relief
“Dispersed” rage, in contrast, lacks a sense of shared project or leadership, of a common understanding of what is wrong and how to put it right. The feeling of dispersed rage is intrinsically frustrating, insofar as it provokes bodily and psychic agitation which it can’t remedy. In this state of mind, we may feel injured or mistreated but can identify neither the source of the injury nor the cure. It’s from this agitated zone of feeling, I suspect, that so many Nextdoor users speak.

Recent events suggest that this raw and undirected kind of anger is prone to manipulation and exploitation, not least by X warriors. The recent riots after the stabbings at a children’s dance class in Southport were largely triggered by online demagogues and provocateurs who spread the false rumour that the suspect, in reality a 17-year-old boy named Axel Rudakubana born in Cardiff to Rwandan parents, was a Muslim immigrant named “Ali al Shakati”.

The former GB News presenter Laurence Fox said the incident was evidence that “We need to permanently remove Islam from Great Britain”, while Nigel Farage, more lethally artful than the pedlars of blatant lies and calumnies, asked whether “the truth” – that the incident was really terror-related – “was being withheld from us”.

Fox will have been fully aware that his objective of the permanent removal of Islam from the UK was as impracticable as it was dehumanising, just as Farage knew that his idle speculations offered no focus or direction for the anger of their intended addressees. For the populist agitator, the aim seems to be not to identify a real injustice and set out the appropriate relief but, on the contrary, to stoke a rage for which there can be no relief, to induce a kind of permanent mass enervation.

Doesn’t this come close to describing the mood of our time? For at least the past decade, and perhaps especially since 2016, with its flashpoints of the Brexit vote and Donald Trump’s election victory, anger has felt like the defining emotional texture of our daily social and political lives, giving rise to a pervasive atmosphere of mutual fear, suspicion and accusation, in which any perception of difference – cultural, ideological, racial, sexual, class – shades quickly into the assumption of enmity.

This public mood has seeped into our private lives and relationships. At the most immediate level, we can point to the well-documented divisions and resentments that Trump, Brexit, Covid restrictions, “the boats”, Gaza and so many other markers of cultural and political alignment have insinuated into the lives of families, friends, couples and communities. In my psychoanalytic consulting room, irritable talk of these sources of division runs alongside more slippery emanations of anger, perceptible in one person’s clipped diction and flared nostrils, in another’s stiff, tightly guarded comportment on the couch, in still another’s coiled, withholding silence.

In her work on psychosomatic disturbances such as addictions and eating disorders, the psychoanalyst Joyce McDougall observed that such patients seem unable to experience feelings. Instead, they are “constantly engaged in immediately dispersing in action” whatever affects them emotionally. Rather than find images or words for their feelings of emptiness and isolation, they discharge their distress in the quick fixes of “medication, food, tobacco, alcohol, opiates and… frantic sexual exploits”.

When nervousness and insecurity become the dominant mood of a society, anger is given licence to spread and grow
McDougall wrote this during the 1980s. The symptoms she was ascribing to relatively circumscribed groups have spread exponentially wider with the advent of the internet and social media. To those compulsions she lists, all of which ultimately exacerbate the feelings of despair they seek to relieve, we can add the permanent itch of provocation and reaction on social media platforms.

Then there are the general conditions of what we might call malign public care: governments and other political actors that manipulate information, democratic institutions and class differences to foment division, fear and mistrust – between “native” citizens and migrants, leavers and remainers, red and blue states, workers and shirkers; internet and TV news media that distort, deny and invent facts in order to stoke the rage of their viewers and listeners; big tech corporations that place us under permanent surveillance and harness our private data to direct our private lives. The psychosomatic patient McDougall describes as “feeling empty, misunderstood” starts to sound eerily like any of us. An angry society starts to seem almost inevitable.

A vandalised mural of Vladimir Putin in Belgrade, Serbia, 2022.
View image in fullscreen
A vandalised mural of Vladimir Putin in Belgrade, Serbia, 2022. Photograph: Marko Đurica/Reuters
When nervousness and insecurity become the dominant mood of a society, anger is given licence to spread and grow. Mistrust has become a dominant feature of our everyday lives, as we find ourselves trapped in informational bubbles, in which disagreement and difference breed not curiosity and exchange but antagonism and mutual cancellation from inside an echo-chamber. It’s from this soil that the grandiose anger of social media warriors grows; the anger of countless virtual citizens trying to cancel out their own felt vulnerabilities and doubts, to brandish a clarity and certainty they crave but can never really achieve.

What makes the anger that drove Brexit and the election of Trump, or Russia’s war on Ukraine, or the current devastating conflict in Israel and Palestine, if not its dogged commitment never to question or examine itself? We assume we’re right so as to evade the risks of others’ – or even our own – curiosity, questions or uncertainty.

It is essential in this context to distinguish anger from aggression, as well as to acknowledge how easily the one shades into the other. Where aggression involves the impulsion to act, to impose oneself on people and things, anger is a feeling. The neuroscientist and philosopher Antonio Damasio distinguishes feelings from emotions, defining the latter as automatic stimulus responses, like freezing in fear or retching in disgust.

Feelings are a way of mapping these reactive responses to produce images and ideas about them. Feelings creatively process what emotions respond to blindly, facilitating what Damasio calls “the possibility of creating novel, non-stereotypical responses”. Anger, in this perspective, is of a higher order than aggression, a transformation of reactive behaviour into a kind of self-reflection.

But anger doesn’t always feel like a triumph of reflective feeling over reactive aggression. There is, after all, a certain satisfaction in the coupling of anger and aggression. Anger sharpens our sense of clarity and righteousness in taking action, whether that means a physical attack, a street protest or a marital row.

The uncoupling of anger from aggression often has the opposite effect. It deprives us of an immediate outlet for action, leaving us with an unrelieved pressure on our nervous system. I’m so angry, we often say, I don’t know what to do with myself. At which point, anger can take us down many different paths. It can lead us to the raw frustration of suppressed rage, or to presenting our anger in the guise of some other attitude (exaggerated politeness, over-friendliness, moroseness). It can also induce us to repress it, to cut ourselves off from the anger we’re feeling.

One of the most basic premises of psychoanalysis, known as transference, is that the patient’s relationship to the analyst is repeating a much older pattern of relating. Clinical work tries to bring the patient to awareness of this tendency, without which they’re likely to perpetuate this repetition instead of resolving it.

In transference, the analyst will come to be experienced, sometimes consciously, more often not, as an avatar of key figures from earlier stages of life: a parent, a teacher, a sibling, a friend, a lover, a colleague or a composite of two or more of these. “You’ll end up totally fed up with me, just like every other man I meet,” a patient may tell me, or “That’s exactly the kind of snarky thing my father would say!” Transference often arouses unruly intensities of feeling in a patient, rendering the analyst an object of love, hate, trust, mistrust, fear, comfort, reverence or contempt, sometimes within a single session.

Underlying these feelings is a profound sense of dependency, derived from the earliest period of life, when our very survival depended on the ministrations of our carers. The basic scenario of psychoanalysis is fraught with power and all its attendant anxieties; a person brings the most vulnerable and hidden region of their psyche and places it in the care of the analyst, in the hope that this gesture of trust won’t be abused or exploited.

A portrait of the psychoanalyst and professor Josh Cohen, a man with brown hair and glasses.
View image in fullscreen
Author Josh Cohen. Photograph: Charlotte Speechley
But the anxiety implied in this hope can never be fully dispelled. What if their show of benignity is a subtly disguised form of control and manipulation? Thought about this way, the risks of the psychoanalytic relationship bear a striking resemblance to the risks of the relationship between citizens and rulers. The erosion of trust in politicians is, we might say, transferential. Citizens are saying, in effect, “if we put ourselves in your hands and trust you to look after our best interests, you’ll only betray us”. It’s this kind of wariness that oils the wheels of the demagogue’s ascent.

The Italian psychoanalyst Massimo Recalcati suggests that in our age of moral chaos and loss of meaning, the younger generation are not so much managing the desire to kill and replace their fathers – as the classical idea of the oedipal complex suggests – than the urgent need for an absent parent to return and restore order and justice.

Recalcati calls this the “Telemachus complex”, referencing the son of Odysseus, who in The Odyssey must endure and hold off the attacks of the menacing Proci invading and usurping his family home, while searching over the horizon for his father to return and right these wrongs.

Doesn’t this state of mind capture precisely the nature of the anger animating the younger generation of climate protesters? Their demand isn’t expressing an urge to kill and usurp the older generation, but a desperate cry across the horizon to the parents who have gone quiet or missing while their planetary home has been violated or ransacked. The anger, embodied for us in the stern ferocity of Greta Thunberg, is directed not towards disposing of the parents but bringing them back to where they are most needed.

Where young climate protesters are using the transference to serve the interests of social and political justice, rightwing populism manipulates the transference to erode autonomy of mind and promote a parody of justice. Trump and Farage take the trust and belief their followers place in them, and their rage against traditional politicians, not to restore justice but to keep their constituencies in a state of permanent anger. Trump’s long campaign of election denial sustains a mass rage that can’t be assuaged. For Trump, anger is the political gift that keeps on giving; his task is to keep it flowing. To achieve redress would risk switching off the tap.

What might happen if we didn’t default to automatic rage at the point we felt personally or politically provoked, if our debate relied less on a repertory of predictable stimulus responses? Some may argue that it would make way for the restoration of a political culture impelled by fact-based reason and the best interests of its citizens.

But after Brexit and Trump, it has become clear that the appeal to facts and best interests is an inadequate basis on which to resist far-right populism. Perhaps it is not so much the rational appeal to facts we need to be making so much as contact with the depth and complexity of our feelings. The politics of “Stop the boats!” and “Build the wall!” feeds off a reactive, defensive rage. Lurking under that coiled anger is a rich and complex seam of emotional experience. Perhaps it is time we started listening to this teeming life of feeling, instead of to the noisy slogans drowning it out.

Josh Cohen is a psychoanalyst and emeritus professor of English at Goldsmiths, University of London. His book All the Rage: Why Anger Drives the World is published on 10 October by Granta (£16.99). To support the Guardian and Observer order your copy at guardianbookshop.com. Delivery charges may apply

 This article was amended on 17 September 2024 to correct the spelling of Charlotte Speechley’s surname.

This is the archive of The Observer up until 21/04/2025. The Observer is now owned and operated by Tortoise Media.

Explore more on these topics
Society
The Observer
The far right
Nigel Farage
Donald Trump
Brexit
Greta Thunberg
Mental health
features
Share
Reuse this content

Most viewed

US military kills two people in strike on alleged drug-trafficking boat in Pacific

US man accused of keeping four adults in basement also faces murder charges

Trump says he has final say on paying himself $230m for past investigations

Democratic senator’s floor speech condemning Trump enters 19th hour

Stephen Colbert on Trump’s White House East Wing demolition: ‘So deeply unsettling’
Most viewed
UKUK politicsEducationMediaSocietyLawScotlandWalesNorthern Ireland
News
Opinion
Sport
Culture
Lifestyle
Original reporting and incisive analysis, direct from the Guardian every morning
Sign up for our email
About us
Help
Complaints & corrections
Contact us
Tip us off
SecureDrop
Privacy policy
Cookie policy
Tax strategy
Terms & conditions
All topics
All writers
Newsletters
Digital newspaper archive
Bluesky
Facebook
Instagram
LinkedIn
Threads
TikTok
YouTube
Advertise with us
Guardian Labs
Search jobs
Work for us
Accessibility settings
 
Back to top
© 2025 Guardian News & Media Limited or its affiliated companies. All rights reserved. (dcr)

"""

# Analyze the article
results = analyze_complete_article(example_article)

# Print results
print_analysis_results(results)

Usage Example:
Number of sentences: 107
FINAL PREDICTIONS:
------------------------------
News Topic: military
   Vote Count: {'education': 2, 'children': 3, 'city-government': 2, 'social-security': 3, 'foreign-policy': 5, 'military': 7, 'animals': 3, 'homeland-security': 2, 'history': 2, 'climate-change': 1, 'public-safety': 3, 'nuclear': 1, 'health-care': 5, 'taxes': 1, 'immigration': 3, 'civil-rights': 5, 'agriculture': 4, 'drugs': 2, 'abortion': 2, 'crime': 3, 'religion': 1, 'deficit': 1, 'corporations': 2, 'marriage': 2, 'florida': 1, 'economy': 2, 'candidates-biography': 1, 'elections': 1, 'afghanistan': 2, 'campaign-finance': 1, 'israel': 1, 'medicare': 1, 'stimulus': 2, 'criminal-justice': 2, 'environment': 5, 'congressional-rules': 1, 'trade': 1, 'federal-budget': 3, 'ethics': 3, 'gays-and-lesbians': 1, 'state-budget': 1, 'voting-record': 1, 'consumer-safety': 1, 'debt': 1, 'job-accomplishments': 2, 'weather': 1, 'housing': 1, 'families': 1, 'obama-birth-certificate': 1, 'gove

In [17]:
# Usage Example
print("📖 Usage Example:")
print("=" * 30)

# Example article
example_article = """
FIRST ON FOX: The White House railed against the "Democrat shutdown" for "jeopardizing national security" because 80% of the federal agency charged with protecting the U.S. nuclear stockpile will be furloughed in the coming days, the administration told Fox News Digital. 

"The Democrat shutdown is now jeopardizing our national security," White House spokeswoman Taylor Rogers told Fox News Digital Friday afternoon. "By refusing to pass the clean, bipartisan funding extension, the Democrats are causing funds to run out for critical programs, resulting in furloughs of personnel at the National Nuclear Security Administration who manage our nuclear stockpile.

"This is reckless and could be completely avoided if the Democrats simply voted to reopen the government and stopped holding the American people hostage.

An administration official confirmed to Fox Digital that 80% of the National Nuclear Security Administration's staff will be furloughed because available funds will soon be expended. 

The National Nuclear Security Administration operates within the U.S. Department of Energy, maintains the nation's nuclear stockpile and works to reduce the threat of nuclear weapons in foreign nations. 

The agency will next enter minimum safe operations, meaning remaining employees will focus on maintaining physical security, cybersecurity, nuclear safety and emergency management, according to an administration official.

"We have not furloughed anyone yet, but we will be out of funds by tomorrow or early next week," Department of Energy Secretary Chris Wright told Bloomberg News Friday of the upcoming furloughs. "So, we will be forced to do that if this shutdown continues.

"We’ve been paying them to date, but, starting tomorrow, Monday at the latest, we’re not going to be able to pay those workers. If that continues on for long, they may get other jobs," Wright told Bloomberg, putting "the sovereignty of the country," at stake.

The administration official told Fox News Digital at there will be significant impacts on the agency's nuclear deterrence mission as various offices shutter during the shutdown, and consequences of the shutdown are expected to last beyond the eventual reopening of the government. 

"As our adversaries build more silos and weapons, we will be turning off the lights," the administration official said.

Republican lawmakers also have sounded off on the upcoming furloughs, including Alabama Rep. Mike Rogers during a House news conference on Friday. 

"We were just informed last night the National Nuclear Security Administration, the group that handles the nuclear stockpile, that the carryover funding they’ve been using is about to run out," he said. "These are not employees that you want to go home. They are managing and handling a very important strategic asset for us. They need to be at work and being paid." 

The U.S. government has been in the midst of an ongoing shutdown since Oct. 1, when Senate lawmakers failed to pass funding legislation for 2026.

The Trump administration and Republicans have since pinned blame for the shutdown on Democrats, claiming they sought taxpayer-funded medical benefits for illegal immigrants. Democrats have denied they want to fund healthcare for illegal immigrants and instead have blamed Republicans for the shutdown.

"Every day that Republicans refuse to negotiate to end this shutdown, the worse it gets for Americans — and the clearer it becomes who’s fighting for them," Senate Minority Leader Chuck Schumer told Fox Digital earlier in October of the shutdown. 

"Each day our case to fix healthcare and end this shutdown gets better and better, stronger and stronger because families are opening their letters showing how high their premiums will climb if Republicans get their way."
"""

# Analyze the article
results = analyze_complete_article(example_article)

# Print results
print_analysis_results(results)

📖 Usage Example:
Number of sentences: 23
FINAL PREDICTIONS:
------------------------------
News Topic: immigration
   Vote Count: {'military': 2, 'trade': 1, 'foreign-policy': 2, 'environment': 1, 'deficit': 1, 'nuclear': 1, 'workers': 1, 'elections': 1, 'transportation': 2, 'consumer-safety': 1, 'city-budget': 1, 'bipartisanship': 1, 'job-accomplishments': 1, 'city-government': 1, 'drugs': 1, 'federal-budget': 1, 'immigration': 3, 'children': 1}

Intent: persuade
   Vote Count: {'persuade': 10, 'inform': 9, 'entertain': 3, 'deceive': 1}

 Sensationalism: sensational
   Vote Count: {'neutral': 10, 'sensational': 13}

 Sentiment: Negative
   Vote Count: {'Positive': 7, 'Neutral': 7, 'Negative': 9}

 Reputation: medium
   Sentence Votes: {'medium': 1}

 Stance: neutral
   Sentence Votes: {'neutral': 1}



# Prediction

## Left

In [18]:
# Usage Example
print("📖 Usage Example:")
print("=" * 30)

# Example article
example_article = """
Monday’s Amazon Web Services outage — and the global disruption it caused — underscored just how reliant the internet has become on a small number of core infrastructure providers.

The ramifications of such outages could only get worse if artificial intelligence becomes as central to work and daily life as tech giants suggest it will in the coming years.

Monday’s outage briefly blocked some people from scheduling doctor’s appointments and accessing banking services. But what if an outage took down the AI tools that doctors were using to help diagnose patients, or that companies used to help facilitate financial transactions?

It may be a hypothetical scenario today, but the tech industry is promising a rapid shift toward AI “agents” doing more work on behalf of humans in the near future – and that could make businesses, schools, hospitals and financial institutions even more reliant on cloud-based services. A global survey of nearly 1,500 firms published by McKinsey & Company in May found that 78% of respondents already use AI in at least one business function, up 55% from a year earlier.

“If there’s an outage and you rely on AI to make your decisions and you can’t access it, that’s going to have an effect on performance,” said Tim DeStefano, associate research professor at Georgetown’s McDonough School of Business.

Monday’s outage had such a widespread impact because many companies rely on cloud providers for the backend functions that support their businesses, such as virtual server space, storage or developer tools. Typically, this set up is more affordable, flexible and secure for those customers, except when AWS experiences an outage. Then it’s effectively a single point of failure for a huge swath of the internet.

To be fair, these services are remarkably sturdy considering the scale of their operations. But outages like Monday’s raise questions about how crucial tech services could be made even more reliable.

AWS serves millions of customers, from retailers and restaurants to financial services firms and government agencies; it holds around 37% of the cloud computing market, according to Gartner. Together with Microsoft and Google, the three companies service around 70% of the market.

And the consolidation of the internet’s backbone is continuing in the age of AI. While there’s some grappling between the big three, Amazon, Microsoft and Google remain by far the prominent cloud computing providers for AI applications, according to Emarketer senior analyst Jacob Bourne — and their futures depend at least in part on serving AI demand.

While websites and apps can still technically function using their companies’ own less powerful on-premises servers, “cloud computing represents a technological prerequisite for using AI,” DeStefano said. That’s because the computers needed to run AI tools are powerful and expensive, and on-site hardware isn’t as easy to modify as business needs change. It just makes more sense to rent that computer space and pay for it only as needed.

And as AI becomes more widespread, data center outages could happen more frequently since AI models are so power-hungry, Bourne said. Major cloud providers, including Amazon, Microsoft and Google, are spending billions on data centers to address this growing need.

The risk of serious disruption from an outage rises considerably the more companies rely on AI agents to do critical tasks and automate the work of humans, a transition that’s already in progress despite disagreement about just how far it will go.

Tech companies are relying on AI to do much of their coding; big banks are hiring fewer workers as they lean more on AI; even Amazon is looking at how AI-enabled robots could automate 75% of its warehouse operations, the New York Times reported Tuesday, citing leaked internal documents. (Amazon says the documents paint a misleading picture of its plans.)

“This is the dream, but if something goes wrong and you don’t have that human intelligence that’s up to speed,” Bourne said, “then we’re really offloading all of these critical tasks to AI and putting a lot of trust in the technology.”

But that threat isn’t inevitable: The shift to AI presents an opportunity to build more resilient internet architecture. Smaller cloud computing competitors like Oracle and CoreWeave are gaining market share with AI-specific offerings. And companies are increasingly relying on multiple cloud providers to create a backstop if one goes down.

Major large language model makers, including Meta and OpenAI, are also investing billions to build their own data centers, which could reduce the strain on shared systems. The tech industry is also pushing to make some AI models smaller and more power-efficient, so they can run locally on smartphones and laptops rather than relying on the cloud so much.

And AI could help find and fix security flaws to prevent outages like Monday’s — if companies invest in those capabilities as much as buzzy, popular tools like AI chatbots and video generation apps.
“There is a pathway to make AI serve us in the best possible ways,” Bourne said. “It doesn’t necessarily seem like we’re on that pathway, though.”"""

# Analyze the article
results = analyze_complete_article(example_article)

# Print results
print_analysis_results(results)

📖 Usage Example:
Number of sentences: 32
FINAL PREDICTIONS:
------------------------------
News Topic: transportation
   Vote Count: {'infrastructure': 1, 'census': 1, 'transportation': 4, 'corporations': 4, 'government-efficiency': 1, 'bankruptcy': 1, 'children': 3, 'agriculture': 1, 'health-care': 1, 'environment': 1, 'abortion': 1, 'economy': 1, 'technology': 3, 'sexuality': 1, 'foreign-policy': 1, 'alcohol': 1, 'immigration': 1, 'homeland-security': 1, 'campaign-advertising': 1, 'florida': 1, 'legal-issues': 1, 'drugs': 1}

Intent: inform
   Vote Count: {'persuade': 10, 'entertain': 4, 'inform': 13, 'deceive': 5}

 Sensationalism: sensational
   Vote Count: {'sensational': 28, 'neutral': 4}

 Sentiment: Positive
   Vote Count: {'Negative': 6, 'Neutral': 8, 'Positive': 18}

 Reputation: medium
   Sentence Votes: {'medium': 1}

 Stance: neutral
   Sentence Votes: {'neutral': 1}



## Right

In [19]:
# Usage Example
print("📖 Usage Example:")
print("=" * 30)

# Example article
example_article = """
 Amazon (AMZN) started off the week racing to fix a massive cloud outage that drove errors for users of apps from McDonald's (MCD) to Robinhood Markets (HOOD). But you wouldn't have known it from looking at Amazon stock earlier this week. Despite the AWS meltdown, Amazon stock is poised to end the week with a modest gain. What's going on?

Shares of the tech giant gained 4.2% across Monday and Tuesday. That marks the best two-day gain for Amazon since early September, MarketSurge data shows. The gains Monday came even as the extent of its Amazon Web Services outage became clear early that morning and set off cascading problems across the broader internet throughout the day.

Amazon stock pulled back in trading Wednesday, but it remains ahead overall for the week. While not as clearly disruptive as the massive CrowdStrike (CRWD) outage from last year, the AWS crash highlighted how the web has become interconnected and reliant on a handful of platforms. That's a concern that some lawmakers and cybersecurity experts have highlighted this week.

But here's the twist: Investors don't seem to be that worried.

"The outage underscored the well-known, inherent vulnerability of cloud computing," Mark Malek, chief investment officer at Siebert Financial, told IBD. "But in this market regime, investors seemed more focused on Amazon's AI-driven growth story than a fleeting technical stumble."

Amazon has suffered major outages before. A prominent one on Dec. 7, 2021, knocked out services across many websites for six hours during the holiday shopping rush. Bloomberg called it "the day the internet paused."

But Amazon stock closed that day nearly 3% higher. And AWS has only grown bigger since. Analysts expect Amazon's cloud revenue will reach $126.8 billion this year, according to FactSet, compared to $62 billion in 2021.

"These outages happen on occasion but they have always been temporary in nature and generally not real disruptive to the company or stock," Eric Clark, chief investment officer at Accuvest Global Advisors, told Investor's Business Daily.

To be sure, Amazon stock was already underperforming the market coming into this week. Shares were hovering just below Amazon's 200-day moving average before Monday's gains. The stock is up just 1% overall this year, the worst performance among the mega-size tech stocks known as the Magnificent Seven.

AWS had already been at the center of the debate around Amazon stock. Shares have mostly slumped since Amazon's Q2 report in late July. The report showed Amazon's cloud growth accelerated at a lesser rate compared to Microsoft (MSFT) and Google parent Alphabet (GOOGL). That reignited concerns that Amazon is falling behind in attracting AI-related business, even as it remains the overall market leader for cloud infrastructure.

Of course, the AWS outage also underlines some of the worries about the tech giant. Outages bring risks of both reputational and financial damage, D.A. Davidson analyst Gil Luria said.

"AWS has a refund guarantee for many customers that will get a refund for the day, which will be a small drag on the December quarter," Luria told IBD. "It will also make life easier for Microsoft Azure and Google Cloud salespeople to sell against AWS for the near future."

Still, Luria added that the "financial impact will likely be small, and it is unlikely many existing customers will leave because of one outage, especially if they are given a refund, since switching costs is so high. Which is why shares have not reacted too negatively."

The main trouble for Amazon originated from an issue with its DynamoDB database service, the company said on the AWS site. The damage was confined to its regional data centers in Virginia. But attempts to resolve the issue led to a range of other problems that left services hosted on AWS buggy for roughly 15 hours.

The company said in its final update on the outage late Monday that all services had returned to normal by 6 p.m. Eastern time that day. The service health page for AWS reported no issues as of Wednesday. Amazon typically publishes larger reports on outages in the weeks following the event.

Other Amazon news may have overshadowed the outage. The New York Times reported Tuesday that internal documents at Amazon show the company believes it can automate 75% of its operations using robotics.

That report likely gave Amazon stock a boost. Robotics has the potential to drive $2 billion to $4 billion in annual savings for Amazon by 2027, analysts with Morgan Stanley said in a client note Wednesday.

On Wednesday, Bloomberg also reported that Anthropic is pursuing a large cloud-computing deal with Google, which also could have impacted Amazon stock. Amazon's investment and partnership with Anthropic are a key part of its AI push.
The next big test for Amazon comes next week. The tech giant will report third-quarter earnings results on Oct. 30.

The big debate on Wall Street is whether Amazon's AWS year-over-year quarterly revenue growth will continue to accelerate to reach 20%. AWS growth has been hovering between 17% and 19% in recent quarters.

"Right now, the most important factors for Amazon's stock are the pace of AWS's recovery and growth, the margin expansion from AI-driven efficiencies, and the company's ability to convert those efficiencies into sustainable free cash flow," Malek said.

Amazon's cloud growth is sure to be compared against rivals Microsoft and Google. Both of those companies are scheduled to report the day before Amazon.

"Consensus seems to think AWS is losing share to Google and Microsoft cloud offerings because they have access to Gemini and OpenAI — we will know in a week or two on earnings," Accuvest's Clark said.

”"""

# Analyze the article
results = analyze_complete_article(example_article)

# Print results
print_analysis_results(results)


📖 Usage Example:
Number of sentences: 54
FINAL PREDICTIONS:
------------------------------
News Topic: elections
   Vote Count: {'sports': 1, 'campaign-finance': 2, 'deficit': 1, 'terrorism': 1, 'elections': 3, 'gas-prices': 2, 'children': 2, 'bankruptcy': 1, 'education': 1, 'civil-rights': 2, 'corrections-and-updates': 1, 'message-machine': 1, 'iraq': 1, 'veterans': 2, 'public-safety': 1, 'families': 1, 'state-budget': 2, 'foreign-policy': 3, 'corporations': 2, 'government-efficiency': 1, 'crime': 2, 'income': 1, 'technology': 3, 'immigration': 2, 'water': 1, 'energy': 1, 'alcohol': 1, 'congress': 1, 'labor': 2, 'history': 1, 'criminal-justice': 1, 'health-care': 2, 'religion': 1, 'trade': 1, 'infrastructure': 1, 'jobs': 1, 'china': 1}

Intent: inform
   Vote Count: {'deceive': 4, 'inform': 32, 'persuade': 14, 'entertain': 4}

 Sensationalism: sensational
   Vote Count: {'sensational': 35, 'neutral': 19}

 Sentiment: Neutral
   Vote Count: {'Neutral': 29, 'Positive': 17, 'Negative': 8

In [20]:
# Usage Example
print("Usage Example:")
print("=" * 30)

# Example article
example_article = """
An Amazon Web Services (AWS) outage on Monday highlighted what experts call the dangers of centralizing the world’s online ecosystem into the hands of a shrinking number of tech companies. Thousands of apps and websites went dark, exposing the fragility of the internet — not just from technical glitches, but also from coordinated cyberattacks.

“Relying on a handful of major cloud providers creates serious vulnerabilities across the internet, putting whole economies and the pillars of civil society at risk,” Raphael Auphan, chief operating officer of the privacy company Proton, told Straight Arrow News. “AWS has become a single point of failure for countless businesses and essential services.”

AWS, which provides web-hosting services and storage space to an estimated 76 million websites, is believed to hold 30% of the world’s cloud infrastructure market share. Microsoft’s Azure controls around 20%, while Google comes in at 13%. In total, the three companies make up more than three-fifths of the market.

The outage, caused by a Domain Name System (DNS) error, affected services such as Slack, Snapchat, Duolingo, Hulu and Grubhub. DNS is often described as the phonebook of the internet, allowing web browsers to load websites by translating URLs, such as Google.com, into machine-readable IP addresses.

The internet’s backbone isn’t just vulnerable to errors. Experts warn that malicious actors could take advantage of the internet’s reliance on major cloud providers.

“It would be relatively straightforward to take a large part of the internet offline, but it would be difficult to disable the internet entirely, as the entire way the fundamental protocols of the internet work is to route around damage,” Harry Halpin, chief executive officer of NymVPN, told SAN. “Yet most of the internet, with the exception of China and Russia, is centralized to a few large cloud providers like Amazon and Google, as well as content delivery networks like Cloudflare and Akamai.

Alexander Chamandy, founder of the information technology company Envescent, also believes “it’s easier than what many may suspect” to severely cripple the internet.

“That is to say, when you have a lack of companies diversifying their cloud presence, you have greater concentration risk,” Chamandy told SAN. “That risk played out during Monday’s outage due to a simple DNS problem internally at Amazon.”

“Could a sophisticated threat actor, such as a foreign government or organized crime group, facilitate a similar outcome?” he said. “Yes. It’s theoretically possible, whether that is through a DDoS (distributed denial of service) attack, breach to cripple critical systems, or even a kinetic attack on infrastructure.”

AWS is used by many critical sectors, including finance, air travel, health care and government agencies. Coinbase, Delta Air Lines, Robinhood, United Airlines and Venmo were just some of the notable services affected on Monday.

Auphan, the COO at Proton, also expressed concern that the major cloud providers are all U.S.-based.

“This outage gave a glimpse of the worst-case scenario and how easily the ripples from an outage can spread globally, with disruptions across health care, banking and transportation, highlighting the danger of our global dependence on U.S. technology,” Auphan said. “Simply put, when the whole world relies on tech from a tiny number of offerings, from one country, then the whole world is vulnerable.

“The only answer for the U.K., Europe and elsewhere,” Auphan said, “is to prioritize digital sovereignty, in other words, to develop their own native services.”

While much of the attention has been placed on the major services taken offline by the outage, Auphan warns that smaller companies, which may not have the financial ability to recover, are much more vulnerable.

“Small and medium-sized businesses, unlike well-resourced corporations, are among the most at risk in these situations,” he said. “Large corporations with teams of engineers, IT specialists and well-funded communications departments can often weather the storm. But when these meltdowns happen, smaller companies that rely on the likes of AWS or Azure simply have to cross their fingers, hope for the best and pray their customers will not walk away.”
”"""

# Analyze the article
results = analyze_complete_article(example_article)

# Print results
print_analysis_results(results)

Usage Example:
Number of sentences: 24
FINAL PREDICTIONS:
------------------------------
News Topic: corporations
   Vote Count: {'corporations': 2, 'science': 1, 'environment': 1, 'corrections-and-updates': 1, 'technology': 1, 'energy': 1, 'transportation': 1, 'candidates-biography': 2, 'legal-issues': 1, 'guns': 1, 'infrastructure': 2, 'china': 1, 'criminal-justice': 1, 'message-machine': 1, 'crime': 1, 'public-safety': 1, 'iraq': 1, 'health-care': 1, 'terrorism': 1, 'city-budget': 1, 'children': 1}

Intent: inform
   Vote Count: {'inform': 15, 'entertain': 2, 'deceive': 3, 'persuade': 4}

 Sensationalism: sensational
   Vote Count: {'sensational': 22, 'neutral': 2}

 Sentiment: Neutral
   Vote Count: {'Neutral': 9, 'Positive': 9, 'Negative': 6}

 Reputation: medium
   Sentence Votes: {'medium': 1}

 Stance: neutral
   Sentence Votes: {'neutral': 1}



# Incorrect 1

In [21]:
# Usage Example
print("Usage Example:")
print("=" * 30)

# Example article
example_article = """
Trump Said His Plans Wouldn’t Touch the White House. Then the East Wing Came Down.
President Trump initially said the ballroom construction would not dismantle parts of the White House. His officials now say it was cheaper and more structurally sound to simply demolish the East Wing.

As roaring machinery tore down one side of the White House, President Trump acknowledged on Wednesday that he was having the entire East Wing demolished to make way for his 90,000-square-foot ballroom, a striking expansion of a project that is remaking the profile of one of the nation’s most iconic buildings.

Mr. Trump was unsentimental as news of the demolition spread. “It was never thought of as being much,” he said of the East Wing, which was home to the first lady’s office and spaces used for ceremonial purposes. “It was a very small building.”

The process of tearing down the East Wing was expected to be completed as soon as this weekend, two senior administration officials said, as Mr. Trump moved rapidly to carry out a passion project that he said was necessary to host state dinners and other events.

But the previously unannounced decision to demolish the East Wing was at odds with Mr. Trump’s previous statements about the project, and underscored his intention to blast through the sensibilities of many in Washington to continue putting a lasting imprint on the White House.


The president also said on Wednesday that the ballroom would cost $300 million, $100 million more than initially estimated.

“In order to do it properly, we had to take down the existing structure,” Mr. Trump said. He also said — somewhat cryptically — that “certain areas are being left.” But the two senior administration officials, who spoke on condition of anonymity to discuss the plans, confirmed that the entire East Wing was being demolished.

The West Wing and the White House residence, where the president lives, are not affected by the project, which is the largest renovation to the White House in decades.

When Mr. Trump first announced his plans for the ballroom, he pledged that the White House would not be touched by the construction.

“It won’t interfere with the current building. It’ll be near it but not touching it,” he said in July. “And pays total respect to the existing building, which I’m the biggest fan of.”


Upon further evaluation, the White House determined it was cheaper and more structurally sound to demolish the East Wing than to build an addition, one of the administration officials said.

On Wednesday, the Secret Service kept onlookers away as heavy machinery ripped away at hunks of the building.

The scope of the demolition, and Mr. Trump’s repeated promises that the White House itself would not be affected by the work, were in many ways symbolic of how he has conducted his presidency. On a variety of issues, Mr. Trump has blown past norms and traditions, often moving so quickly that it can be too late for courts, Congress or the public to catch up.

The planned size of the ballroom would transform the footprint of the White House campus. At 90,000 square feet, the ballroom would be nearly double the size of the White House residence, which is 55,000 square feet.

The ballroom is only the latest renovation plan that Mr. Trump has undertaken since he took office for the second time. He is also leaving his mark on the Oval Office, which now features many gilded flourishes. He also paved over the Rose Garden; erected huge flag poles on the White House grounds; and is planning to build an arch in front of Arlington National Cemetery in the style of the Arc de Triomphe.


Sara C. Bronin, a law professor at George Washington University who led the Advisory Council on Historic Preservation under former President Joseph R. Biden Jr., said that Mr. Trump’s decision to tear down the East Wing appeared to run afoul of the National Historic Preservation Act, which requires federal agencies to take into account the effects of their actions on historic places.

“The Trump administration’s shortsighted decision to start demolishing parts of the White House is exactly the kind of action the N.H.P.A. was passed to circumvent,” she said.

Mr. Trump has said that he is raising tens of millions of dollars in private donations to fund the project. The president plans to contribute some of his own money as well, though the amount has not been determined, one of the officials said.

“It’s being paid for 100 percent by me and some friends of mine,” Mr. Trump said.

The East Wing was built in 1902 during Theodore Roosevelt’s presidency as an extension to the White House, but was overhauled in the 1940s at the request of President Franklin D. Roosevelt.

It was built primarily to conceal an underground bunker, the Presidential Emergency Operations Center. It also added formal work space for the White House staff, including the offices of the first lady. Lorenzo Winslow was the architect of that addition.


It has also housed the White House Social Office and served as a headquarters for planning parties, state dinners and other events.

With the demolition of the East Wing goes a slice of history. It was where President Bill Clinton met secretly with Dick Morris, a political adviser, without his staff knowing. It was where Vice President Dick Cheney was hustled to a bunker after the Sept. 11 terrorist attacks. Mr. Trump was rushed there, too, during protests in 2020.

The White House on Tuesday did not answer questions about the bunker, but one of the administration officials said the new structure would also have enhanced security features.

Amid backlash to the demolition, the White House has defended its decision, publishing photos of past renovations and construction projects undertaken by presidents.

The White House says the ballroom, once finished, will have a seated capacity of 650 people, though Mr. Trump said recently it would hold 999 people.
"""

# Analyze the article
results = analyze_complete_article(example_article)

# Print results
print_analysis_results(results)

Usage Example:
Number of sentences: 43
FINAL PREDICTIONS:
------------------------------
News Topic: bipartisanship
   Vote Count: {'bipartisanship': 4, 'corporations': 3, 'diversity': 1, 'iraq': 3, 'state-budget': 3, 'obama-birth-certificate': 2, 'military': 2, 'campaign-finance': 1, 'ebola': 1, 'crime': 1, 'congress': 2, 'history': 3, 'unions': 1, 'abortion': 1, 'county-government': 1, 'guns': 2, 'environment': 1, 'job-accomplishments': 1, 'county-budget': 1, 'congressional-rules': 1, 'elections': 1, 'terrorism': 2, 'ethics': 2, 'china': 1, 'trade': 1, 'message-machine-2012': 1}

Intent: inform
   Vote Count: {'inform': 25, 'deceive': 4, 'entertain': 6, 'persuade': 8}

 Sensationalism: sensational
   Vote Count: {'sensational': 36, 'neutral': 7}

 Sentiment: Neutral
   Vote Count: {'Neutral': 19, 'Negative': 7, 'Positive': 17}

 Reputation: medium
   Sentence Votes: {'medium': 1}

 Stance: neutral
   Sentence Votes: {'neutral': 1}

